# LGMD — Language-Guided Matrix Decomposition

Faithful re-implementation of *Interpretable Concept Discovery via Language-Guided Matrix Decomposition* (ECCV 2026 #11398).

Scope: a single ImageNet-1k class (**tabby cat**, n02123045). Backbone defaults to **ResNet34** (paper); switch to **MobileNetV2** / ResNet50 via `CONFIG["backbone"]`. Baselines (paper Sec 4: **ICE, CRAFT, FACE**) and the reported metrics (**Acc**, **C-Ins**, MSE) are included. Concepts are read from the stored `concept_vocab.json` table (keyed by class name) and CLIP-filtered. Constant artifacts are cached to Google Drive on Colab.

In [ ]:
# === Colab setup (skip when running locally) ===
# 1) Clone the repo from GitHub and install dependencies:
# !git clone https://github.com/<you>/lgmd.git
# %cd lgmd
# !pip -q install -r requirements.txt
#
# 2) Add secrets in the Colab 'Secrets' panel (key icon): HF_TOKEN.
#    (HF_TOKEN must have accepted the gated ImageNet-1k terms.)
# 3) Importing config below mounts Google Drive automatically when on Colab.

In [ ]:
import os, json, sys

# This notebook belongs to the lgmd repo. A sibling repo (sparse_concept_discovery)
# is cloned alongside it and ALSO has a src/config.py, so we anchor on a marker unique
# to this repo (src/lgmd.py) to avoid importing the wrong config. We locate the repo
# root regardless of where the kernel started, cd into it (so config's REPO_ROOT-relative
# paths resolve), and put its src/ first on sys.path.
_MARKERS = (os.path.join("src", "config.py"), os.path.join("src", "lgmd.py"))
_here = os.getcwd()
_candidates = [
    _here,
    os.path.join(_here, "lgmd_concept_discovery"),
    "/content/lgmd_concept_discovery",
    os.path.dirname(_here),
]
for _root in _candidates:
    if all(os.path.isfile(os.path.join(_root, m)) for m in _MARKERS):
        os.chdir(_root)
        sys.path.insert(0, os.path.join(_root, "src"))
        break
else:
    raise RuntimeError(
        "Could not locate the lgmd repo root (needs src/config.py and src/lgmd.py). "
        "On Colab, %cd /content/lgmd_concept_discovery first."
    )

import torch
from config import CONFIG, cache_name
import utils, data_utils, model_utils, concepts as concept_mod, clip_maps, lgmd, baselines, metrics, viz

utils.set_seed(CONFIG["seed"])
DEVICE = model_utils.DEVICE
CDIR = CONFIG["cache_dir"]       # heavy/constant artifacts (activations, S, W) on Drive
RDIR = CONFIG["results_dir"]     # cached metric tables on Drive
bb = CONFIG["backbone"]          # used in figure filenames
print("device:", DEVICE)

## 1. Data — one ImageNet-1k class

In [ ]:
images = data_utils.load_class_images(CONFIG["n_train"] + CONFIG["n_val"])
train_imgs, val_imgs = data_utils.make_splits(images, CONFIG["n_train"], CONFIG["n_val"])
print(f"train: {len(train_imgs)}  val: {len(val_imgs)}")

## 2. Backbone + encoder activations (cached to Drive)

In [ ]:
model, transform = model_utils.load_backbone()
Z_train = utils.cached(os.path.join(CDIR, cache_name("Z_train", ".pt", "data", "model")),
    lambda: model_utils.extract_activations(model, transform, train_imgs, desc="train activations"))
Z_val = utils.cached(os.path.join(CDIR, cache_name("Z_val", ".pt", "data", "model")),
    lambda: model_utils.extract_activations(model, transform, val_imgs, desc="val activations"))
A_train, A_val = lgmd.unfold(Z_train), lgmd.unfold(Z_val)
print(bb, "| Z_train:", tuple(Z_train.shape), " A_train:", tuple(A_train.shape))

## 3. Concept vocabulary (stored table) + CLIP filtering

In [ ]:
clip = clip_maps.CLIP()
# Pass train_imgs so stage-2 filtering ranks concepts by CLIP relevance to the
# class image prototype before diversity dedup (suppl. A1.4).
concept_list = concept_mod.get_concepts(clip, images=train_imgs)
assert len(concept_list) == CONFIG["r"], len(concept_list)
print(concept_list)

## 4. Localized CLIP similarity maps S (cached — most expensive stage)

In [ ]:
S_train = utils.cached(os.path.join(CDIR, cache_name("S_train", ".pt", "data", "con", "clip")),
    lambda: clip_maps.build_S(train_imgs, concept_list, clip))
print("S_train:", tuple(S_train.shape))

## 5. Fit the semantic concept basis W (PGD)

In [ ]:
W = utils.cached(os.path.join(CDIR, cache_name("W", ".pt", "data", "model", "con", "clip", "pgd")),
    lambda: lgmd.fit_basis(A_train, S_train))
print("W:", tuple(W.shape))

## 6. Inference on val + predictive preservation / faithfulness

In [ ]:
label = CONFIG["class_index"]

# Paper Sec 4: concept analysis is performed on correctly classified validation samples.
orig_logits_full = model_utils.logits_from_Z(model, Z_val)
keep = orig_logits_full.argmax(-1) == label
print(f"correctly classified val: {int(keep.sum())}/{len(keep)}")
Z_val = Z_val[keep]
val_imgs = [im for im, k in zip(val_imgs, keep.tolist()) if k]
A_val = lgmd.unfold(Z_val)
orig_logits = orig_logits_full[keep]

S_hat = lgmd.infer(A_val, W)                     # closed-form + PGD (Eqs. 4-5)
A_hat = lgmd.reconstruct(S_hat, W, Z_val.shape)

def _lgmd_metrics():
    recon_logits = model_utils.logits_from_Z(model, A_hat)
    return {
        **metrics.predictive_preservation(orig_logits, recon_logits, label),
        "kl": metrics.kl_logits(orig_logits, recon_logits),
        "recon_err": metrics.recon_error(A_val, lgmd.unfold(A_hat)),
    }

lgmd_metrics = utils.cached_json(
    os.path.join(RDIR, cache_name("lgmd_metrics", ".json", "data", "model", "con", "clip", "pgd", "infer")),
    _lgmd_metrics)
print("LGMD:", lgmd_metrics)

## 7. Baseline comparison (ICE, CRAFT, FACE vs LGMD) — Acc + C-Ins

In [ ]:
R = CONFIG["r"]
face_head = lambda a: model_utils.classify_pooled(model, a.to(DEVICE))  # grad-enabled, arch-aware
head_fn = lambda Z: model_utils.logits_from_Z(model, Z)

# Paper Sec 4 baselines: ICE, CRAFT, FACE (+ ours). Identical pipeline / splits / r;
# only the learned basis differs. The (expensive) baseline fits are cached on Drive.
def _comparison():
    bases = {
        "ICE":   baselines.fit_ice(A_train, R),
        "CRAFT": baselines.fit_craft(A_train, R),
        "FACE":  baselines.fit_face(A_train, R, face_head, Z_train.shape),
        "LGMD":  W,
    }
    out = {}
    for name, Wb in bases.items():
        Sb = lgmd.infer(A_val, Wb)
        Ab = lgmd.reconstruct(Sb, Wb, Z_val.shape)
        lg = model_utils.logits_from_Z(model, Ab)
        cur = metrics.faithfulness_curves(Sb, Wb, Z_val.shape, head_fn, label)
        pp = metrics.predictive_preservation(orig_logits, lg, label)
        out[name] = {
            "Acc": pp["recon_acc"],                            # predictive preservation
            "C-Ins": metrics.insertion_auc(cur["insertion"]),  # concept importance
            "agreement": pp["agreement"],
            "kl": metrics.kl_logits(orig_logits, lg),
            "recon_err": metrics.recon_error(A_val, lgmd.unfold(Ab)),
        }
    return out

rows = utils.cached_json(
    os.path.join(RDIR, cache_name("comparison", ".json",
                                  "data", "model", "con", "clip", "pgd", "infer", "base", "cins")),
    _comparison)
print(json.dumps(rows, indent=2))

## 8. Faithfulness curves (C-Deletion / C-Insertion) for LGMD

In [ ]:
import matplotlib.pyplot as plt
curves = metrics.faithfulness_curves(S_hat, W, Z_val.shape, head_fn, label)
plt.plot(curves["deletion"], label="C-Deletion")
plt.plot(curves["insertion"], label="C-Insertion")
plt.xlabel("# concepts removed / added"); plt.ylabel("true-class prob"); plt.legend()
plt.savefig(os.path.join(RDIR, f"faithfulness_{bb}.png"), dpi=120, bbox_inches="tight")
plt.show()

## 9. Concept heatmap overlays (Eq. 6)

In [ ]:
viz.save_concept_overlays(val_imgs, S_hat, concept_list, out_name=f"overlays_{bb}.png", max_images=4)

## 10. Ablation — closed-form vs closed-form+PGD coefficient estimation (Table 4)

In [ ]:
# Closed-form (Eq. 4) vs closed-form + PGD (Eq. 5). Both preserve accuracy; the
# hybrid slightly improves C-Ins and lowers reconstruction MSE (paper Sec 4.5). Cached on Drive.
def _ablation():
    out = {}
    for mode, refine in [("Closed-form", False), ("Closed-form+PGD", True)]:
        Sh = lgmd.infer(A_val, W, refine=refine)
        Ah = lgmd.reconstruct(Sh, W, Z_val.shape)
        lg = model_utils.logits_from_Z(model, Ah)
        pp = metrics.predictive_preservation(orig_logits, lg, label)
        cur = metrics.faithfulness_curves(Sh, W, Z_val.shape, head_fn, label)
        out[mode] = {
            "Acc": pp["recon_acc"],
            "C-Ins": metrics.insertion_auc(cur["insertion"]),
            "MSE": metrics.mse(A_val, lgmd.unfold(Ah)),
        }
    return out

abl = utils.cached_json(
    os.path.join(RDIR, cache_name("ablation_table4", ".json",
                                  "data", "model", "con", "clip", "pgd", "infer", "cins")),
    _ablation)
print(json.dumps(abl, indent=2))

## 11. Concept activation scores: before (CLIP init) vs after optimization (Fig 4)

In [ ]:
# "Before" = raw CLIP-similarity init on val images (expensive: cached); "after" = S_hat.
# Depends on backbone too: val images are filtered to correctly classified samples.
S_val_clip = utils.cached(os.path.join(CDIR, cache_name("S_val", ".pt", "data", "model", "con", "clip")),
    lambda: clip_maps.build_S(val_imgs, concept_list, clip))   # same row order as A_val / S_hat
viz.plot_score_distributions(S_val_clip, S_hat, out_name=f"fig4_score_distributions_{bb}.png")